# CHORUS Vision Physics — UDT / AOR / VOH

**Joseph Black & Connor White** · differential-harness · June 2026

Full physics design for the vision stack:
- **UDT** — Universal Differential Tink (rays, particle bytes, k_m,eff)
- **AOR** — Acoustic-Osmotic Ram (resonance + brine motor + ram pipe)
- **VOH** — Vortex-Osmotic Hydro / Z-Hydro (spin + z-leg)

Specs: `docs/VISION.md`, `docs/UDT_PHYSICS.md`, `docs/AOR_PHYSICS.md`, `docs/VOH_PHYSICS.md`

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from simulation.constants import (
    C_BRINE_8PCT, C_SOUND_WATER, C_TREATED_WW, F_US_DEFAULT,
    I_NACL, R_GAS, T_REF,
)
from simulation.pro_cycle import steady_state_pro
from simulation.parasitics import skid_energy_balance
from simulation.differential_tink import (
    lambda_e90, ray_field, particle_bytes, tink_kernel, udt_pro_state, sweep_eta_tink,
)
from simulation.acoustic_osmotic_ram import aor_state, sweep_column_height
from simulation.vortex_osmotic_hydro import breakeven_omega, sweep_omega, voh_state
from simulation.membrane_transport import concentration_polarization_profile

## 1. PRO baseline (Layer F)

In [ ]:
st = steady_state_pro(C_BRINE_8PCT, C_TREATED_WW, 0.72)
bal = skid_energy_balance(st)
print(f'Δπ = {st.delta_pi/1e6:.3f} MPa')
print(f'ΔP* = {st.delta_P_star/1e5:.1f} bar')
print(f'P_PRO = {st.P_elec_equiv_W:.3f} W')
print(f'P_net (with PX) = {bal.P_net_W:.3f} W')

## 2. UDT — ray field, bytes, Tink kernel

$$k_{m,\mathrm{eff}} = k_{m,0}(1 + \eta_{\mathrm{tink}} \bar{w})$$

$$\mathrm{byte\_len} \propto \frac{A_{\mathrm{loop}}}{L_{\mathrm{line}}} \frac{\lambda_{e90}}{\lambda_0}$$

In [ ]:
lam = lambda_e90()
print(f'λ_e90 @ {F_US_DEFAULT/1000:.0f} kHz = {lam:.4f} m')

rays = ray_field(n_rays=64, coherent_phases=True, seed=1)
bytes_ = particle_bytes(rays, A_loop_m2=0.72, L_line_m=0.3)
tink = tink_kernel(rays, bytes_, k_m0=1.5e-5, eta_tink=0.2)
print(f'flux_gain = {tink.flux_gain:.3f}')
print(f'k_m,eff = {tink.k_m_eff_m_s:.2e} m/s')
print(f'P_actuation = {tink.P_actuation_W:.4f} W')

st_udt, tink2, us = udt_pro_state(eta_tink=0.2)
print(f'UDT P_loop = {st_udt.P_elec_equiv_W:.3f} W, US net gain = {us.P_net_gain_W:.4f} W')

In [ ]:
eta_sweep = sweep_eta_tink(21)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot([r['eta_tink'] for r in eta_sweep], [r['flux_gain'] for r in eta_sweep], 'o-', label='flux_gain')
ax2 = ax.twinx()
ax2.plot([r['eta_tink'] for r in eta_sweep], [r['P_net_gain_W'] for r in eta_sweep], 's--', color='crimson', label='P_net gain')
ax.set_xlabel('η_tink')
ax.set_ylabel('flux gain')
ax2.set_ylabel('P_net gain (W)')
ax.set_title('E14: UDT η_tink sweep')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. AOR — Acoustic-Osmotic Ram

Resonant column: $f_{res} \approx c/(4H)$ · Brine motor: $\Delta\pi$ · Ram leg: Bernoulli momentum

In [ ]:
aor = aor_state()
print(f'f_res = {aor.resonant.f_us_Hz:.0f} Hz, column H = {aor.resonant.height_m:.2f} m')
print(f'P_osmotic = {aor.P_osmotic_MPa:.2f} MPa')
print(f'P_PRO = {aor.P_pro_W:.3f} W, P_net = {aor.P_net_W:.3f} W')

col_sweep = sweep_column_height()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot([r['height_m'] for r in col_sweep], [r['P_net_W'] for r in col_sweep], 's-', color='#dd6b20')
ax.set_xlabel('Column height H (m)')
ax.set_ylabel('P_net (W)')
ax.set_title('E15: AOR resonant column')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. VOH — Vortex-Osmotic Hydro / Z-Hydro

$$P(r,z) = P_0 + \rho g z + \tfrac{1}{2}\rho\omega^2 r^2 + \Pi_{osm}$$

In [ ]:
omega_sweep = sweep_omega(omega_values=list(np.linspace(0, 150, 31)))
flat = voh_state(omega_rad_s=0.0, delta_h_z_m=0.0)
be = breakeven_omega()
print('Flat P_net:', flat.P_net_W, 'W')
print('Breakeven:', be)

fig, ax = plt.subplots(figsize=(8, 4))
rpm = [p['rpm'] for p in omega_sweep]
P_net = [p['P_net_W'] for p in omega_sweep]
ax.plot(rpm, P_net, 'o-', color='#2b6cb0', label='VOH P_net(ω)')
ax.axhline(flat.P_net_W, color='crimson', ls='--', label=f'Flat ({flat.P_net_W:.2f} W)')
if be:
    ax.axvline(be['rpm'], color='green', ls=':', label=f"Breakeven {be['rpm']:.0f} RPM")
ax.set_xlabel('RPM')
ax.set_ylabel('P_net (W)')
ax.set_title('E16: VOH spin vs no-spin — milestone #3')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = ROOT / 'exports' / 'figures' / 'fig16_voh_omega_nb.png'
fig.savefig(out, dpi=200)
print('Saved', out)
plt.show()

## 5. Full stack waterfall

In [ ]:
st_udt, tink, us = udt_pro_state(eta_tink=0.2)
aor = aor_state()
voh = voh_state(omega_rad_s=be['omega_rad_s'] if be else 75.0)

labels = ['PRO baseline', 'UDT+g', 'AOR P_net', 'VOH P_net']
vals = [st.P_elec_equiv_W, st_udt.P_elec_equiv_W, aor.P_net_W, voh.P_net_W]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(labels, vals, color=['#2c5282', '#805ad5', '#dd6b20', '#2b6cb0'])
ax.set_ylabel('W')
ax.set_title('Vision stack power progression (simulation)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Export vision summary JSON

In [ ]:
from simulation.experiments import vision_stack_experiments
vs = vision_stack_experiments()
out = ROOT / 'exports' / 'vision_physics_summary.json'
out.write_text(json.dumps(vs, indent=2), encoding='utf-8')
print('Wrote', out)
print('Summary:', vs['summary'])